# Pipeline RAG — Liquidación de Siniestros de Salud (Caso Zurich Chile + LISA Insurtech)

**Asignatura:** ISY0101 — Ingeniería de Soluciones con IA · DUOC UC
**Integrantes:** Jussara Loza · Roberto Bustamante

Prototipo de pipeline RAG con **Google Gemini API** + **LangChain** + **FAISS**:
corpus de pólizas y reglas → chunking → embeddings → búsqueda vectorial → respuesta con LLM (RAG y Chain-of-Thought) → evaluación con métricas.

## Antes de ejecutar
1. Obtén una API key gratuita en **https://aistudio.google.com/apikey**.
2. En Colab: menú lateral **🔑 Secretos** → *Agregar nuevo secret* → nombre `GEMINI_API_KEY` → pega la clave → activa **Acceso al notebook**.
3. Ejecuta las celdas **en orden** (o *Entorno de ejecución → Ejecutar todas*).

## Celda 1 — Instalación de dependencias
*Evidencia: `01_instalacion.png`*

In [ ]:
# ============================================================
# CELDA 1: Instalación de dependencias para Gemini
# ============================================================
# pandas, numpy y matplotlib ya vienen en Colab: NO se reinstalan
!pip install -q langchain-core langchain-google-genai langchain-community langchain-text-splitters faiss-cpu

import importlib.metadata as md
print("📦 Versiones instaladas:")
for paquete in ["langchain-core", "langchain-google-genai", "langchain-community",
                "langchain-text-splitters", "faiss-cpu", "pandas", "numpy"]:
    try:
        print(f"   {paquete}: {md.version(paquete)}")
    except md.PackageNotFoundError:
        print(f"   {paquete}: ✗ NO INSTALADO")

print("✅ Dependencias instaladas correctamente")

## Celda 2 — Configuración de credenciales
*Evidencia: `02_credenciales.png`*

In [ ]:
# ============================================================
# CELDA 2: Configuración de credenciales desde Secrets de Colab
# ============================================================
from google.colab import userdata
import os

try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception as e:
    api_key = None
    print(f"❌ No se pudo leer el secret GEMINI_API_KEY: {e}")
    print("   Revisa: nombre exacto 'GEMINI_API_KEY' y toggle 'Acceso al notebook' activado")

if api_key:
    os.environ["GOOGLE_API_KEY"] = api_key
    print("✅ Credenciales cargadas desde Secrets de Colab")
    print(f"   API Key: ✓ Configurada ({api_key[:6]}...{api_key[-4:]})")
else:
    print("   API Key: ✗ No configurada")

## Celda 3 — Inicialización del cliente Gemini
Detecta automáticamente qué modelos tiene disponibles la API key y usa el primero que responda.

*Evidencia: `03_cliente_gemini.png`*

In [ ]:
# ============================================================
# CELDA 3: Inicialización del cliente Gemini (detección automática de modelo)
# ============================================================
import os
import time
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI

PAUSA_ENTRE_LLAMADAS = 4  # segundos, para no superar el límite gratuito

# Orden de preferencia (se usan solo los que tu API key tenga disponibles)
PREFERIDOS = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-flash-latest",
              "gemini-flash-lite-latest", "gemini-2.0-flash", "gemini-2.0-flash-lite"]
EXCLUIR = ["tts", "image", "live", "audio", "embedding", "thinking", "exp"]

# --- 1. Verificar API key ---
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("❌ No hay API key cargada. Ejecuta primero la CELDA 2.")

# --- 2. Consultar qué modelos tiene disponibles tu API key ---
cliente = genai.Client(api_key=api_key)
disponibles = []
for m in cliente.models.list():
    acciones = getattr(m, "supported_actions", None) or []
    nombre = m.name.replace("models/", "")
    if "generateContent" in acciones and "gemini" in nombre and not any(x in nombre for x in EXCLUIR):
        disponibles.append(nombre)

print(f"📋 Modelos de texto disponibles para tu API key: {len(disponibles)}")
for n in disponibles[:15]:
    print(f"   - {n}")

# Primero los preferidos (en orden), luego cualquier otro 'flash' disponible
candidatos = [p for p in PREFERIDOS if p in disponibles]
candidatos += [n for n in disponibles if "flash" in n and n not in candidatos]
if not candidatos:
    candidatos = disponibles[:3]

# --- 3. Funciones de apoyo ---
def texto_de(respuesta):
    """Extrae el texto de la respuesta de Gemini (puede venir como str o como lista)."""
    contenido = respuesta.content
    if isinstance(contenido, str):
        return contenido
    partes = []
    for p in contenido:
        if isinstance(p, str):
            partes.append(p)
        elif isinstance(p, dict) and p.get("type") == "text":
            partes.append(p.get("text", ""))
    return "".join(partes)

def es_limite_de_uso(error):
    msg = str(error)
    return any(x in msg for x in ["429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"]) or "quota" in msg.lower()

def invocar_llm(prompt, reintentos=5, espera=15):
    """Llama al LLM con reintentos automáticos si se alcanza el límite de uso."""
    if llm is None:
        raise RuntimeError("❌ El LLM no está inicializado. Vuelve a ejecutar la CELDA 3.")
    for intento in range(1, reintentos + 1):
        try:
            return texto_de(llm.invoke(prompt)).strip()
        except Exception as e:
            if es_limite_de_uso(e):
                print(f"   ⏳ Límite de uso alcanzado, esperando {espera}s (intento {intento}/{reintentos})")
                time.sleep(espera)
                espera *= 2
            else:
                raise
    raise RuntimeError("❌ Gemini no respondió tras varios reintentos. Espera unos minutos y reintenta.")

# --- 4. Probar modelos hasta que uno responda ---
llm = None
MODELO_LLM = None
errores = {}

for modelo in candidatos:
    candidato = ChatGoogleGenerativeAI(model=modelo, temperature=0.1, max_retries=1)
    for intento in range(1, 3):
        try:
            prueba = texto_de(candidato.invoke("Responde solo: OK")).strip()
            llm, MODELO_LLM = candidato, modelo
            break
        except Exception as e:
            errores[modelo] = str(e)[:250]
            if es_limite_de_uso(e) and intento == 1:
                print(f"   ⏳ {modelo}: límite de uso, reintentando en 20s...")
                time.sleep(20)
            else:
                break
    if llm is not None:
        break
    print(f"⚠️ {modelo} no respondió: {errores[modelo]}")

if llm is None:
    raise RuntimeError(
        "❌ Ningún modelo respondió.\n"
        "   - Si el error dice 'quota' o 'limit: 0': ese modelo no tiene cuota gratis para tu cuenta\n"
        "     o la agotaste por hoy. Espera unos minutos o crea una API key nueva en AI Studio.\n"
        "   - Si dice 'API key not valid': revisa el secret GEMINI_API_KEY (CELDA 2)."
    )

print("\n✅ Cliente Gemini inicializado y conectado")
print(f"   Modelo: {MODELO_LLM}")
print(f"   Respuesta de prueba: {prueba}")

## Celda 4 — Corpus documental (base de conocimiento simulada)
*Evidencia: `04_corpus.png`*

In [ ]:
# ============================================================
# CELDA 4: Corpus documental simulado (base de conocimiento)
# ============================================================
documentos = [
    {"id": "POL-01", "tipo": "poliza",
     "texto": "Póliza de salud Zurich Plan Premium. Cobertura de consultas médicas de especialistas hasta $100.000 por evento. Carencia de 30 días desde la contratación. No cubre tratamientos estéticos ni experimentales."},
    {"id": "POL-02", "tipo": "poliza",
     "texto": "Póliza de salud Zurich Plan Premium. Cobertura de exámenes de laboratorio hasta $50.000 por evento. Requiere orden médica vigente. Carencia de 15 días."},
    {"id": "POL-03", "tipo": "poliza",
     "texto": "Póliza de salud Zurich Plan Premium. Cobertura de hospitalizaciones hasta $500.000 por evento. Sujeto a deducible de $50.000. Requiere autorización previa para procedimientos programados."},
    {"id": "REG-01", "tipo": "regla",
     "texto": "Regla de negocio: Toda factura médica debe incluir RUT del prestador, fecha de emisión y detalle de prestación. Facturas sin RUT son rechazadas automáticamente."},
    {"id": "REG-02", "tipo": "regla",
     "texto": "Regla de negocio: Si el monto solicitado excede el límite de cobertura en más del 20%, el caso se deriva a revisión manual."},
    {"id": "REG-03", "tipo": "regla",
     "texto": "Regla de negocio: Siniestros presentados después de 60 días desde la fecha del servicio requieren justificación escrita del asegurado."},
    {"id": "FRA-01", "tipo": "fraude",
     "texto": "Patrón de fraude: Múltiples siniestros con montos idénticos presentados en menos de 7 días por el mismo asegurado deben marcarse como sospechosos."},
    {"id": "FRA-02", "tipo": "fraude",
     "texto": "Patrón de fraude: Facturas con RUT de prestador no registrado en el directorio oficial deben marcarse para verificación."},
]

print(f"✅ Corpus cargado: {len(documentos)} documentos")
for d in documentos:
    print(f"[{d['id']}] ({d['tipo']}) {d['texto'][:80]}...")

## Celda 5 — Chunking
*Evidencia: `05_chunking.png`*

In [ ]:
# ============================================================
# CELDA 5: Chunking con RecursiveCharacterTextSplitter
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Cada documento se divide por separado y conserva sus metadatos (id y tipo)
docs_base = [
    Document(page_content=d["texto"], metadata={"id": d["id"], "tipo": d["tipo"]})
    for d in documentos
]
chunks = splitter.split_documents(docs_base)

total_caracteres = sum(len(d["texto"]) for d in documentos)
print("✅ Chunking completado")
print(f"   Texto original: {total_caracteres} caracteres")
print(f"   Chunks generados: {len(chunks)}")
print(f"   Tamaño promedio: {sum(len(c.page_content) for c in chunks) // len(chunks)} caracteres\n")

for i, c in enumerate(chunks, 1):
    print(f"Chunk {i} [{c.metadata['id']}] ({len(c.page_content)} car.): {c.page_content[:70]}...")

## Celda 6 — Embeddings + FAISS
*Evidencia: `06_embeddings_faiss.png`*

In [ ]:
# ============================================================
# CELDA 6: Embeddings + FAISS Vector Store con Gemini
# ============================================================
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_db = FAISS.from_documents(chunks, embeddings)

print("✅ Base vectorial FAISS creada")
print(f"   Chunks indexados: {vector_db.index.ntotal}")
print(f"   Dimensión de los vectores: {vector_db.index.d}")
print("   Modelo de embeddings: gemini-embedding-001")

# Prueba de búsqueda
query_test = "¿Cuál es la cobertura de consultas médicas?"
resultados_busqueda = vector_db.similarity_search_with_score(query_test, k=2)
print(f"\n🔍 Prueba de búsqueda: '{query_test}'")
for i, (doc, score) in enumerate(resultados_busqueda, 1):
    print(f"\nResultado {i} [{doc.metadata['id']}] (distancia: {score:.4f}):")
    print(f"   {doc.page_content[:120]}...")

## Celda 7 — Cadena RAG básica
*Evidencia: `07_rag_basico.png`*

In [ ]:
# ============================================================
# CELDA 7: Cadena RAG (recuperación + generación)
# ============================================================
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

PLANTILLA_RAG = """Eres un asistente de liquidación de siniestros de salud de Zurich.
Responde SOLO con la información del contexto. Si el contexto no alcanza, dilo.

Contexto:
{contexto}

Pregunta: {pregunta}

Responde en español, en máximo 4 líneas, indicando una decisión
(APROBAR / RECHAZAR / REQUIERE_REVISION_MANUAL) y su motivo."""

def consultar(query):
    docs = retriever.invoke(query)
    contexto = "\n".join(f"- {d.page_content}" for d in docs)
    respuesta = invocar_llm(PLANTILLA_RAG.format(contexto=contexto, pregunta=query))
    return {
        "respuesta": respuesta,
        "contexto": contexto,
        "fuentes": [d.metadata["id"] for d in docs],
    }

# Prueba
print("=" * 60)
print("PRUEBA RAG: Cobertura de consulta cardiológica")
print("=" * 60)
q1 = "¿Está cubierta una consulta cardiológica de $85.000?"
r1 = consultar(q1)
print(f"❓ Pregunta: {q1}")
print(f"\n🤖 Respuesta:\n{r1['respuesta']}")
print(f"\n📚 Fuentes recuperadas ({len(r1['fuentes'])}): {', '.join(r1['fuentes'])}")

## Celda 8 — Prompt Chain-of-Thought + RAG
*Evidencia: `08_prompt_cot.png`*

In [ ]:
# ============================================================
# CELDA 8: Prompt Chain-of-Thought + RAG
# ============================================================
def formato_clp(monto):
    return f"${monto:,}".replace(",", ".")

def decision_con_cot(prestacion, monto, fecha):
    # Recuperar contexto relevante (k=4 para incluir reglas de negocio)
    docs = vector_db.similarity_search(f"{prestacion} cobertura límite monto", k=4)
    contexto = "\n".join(f"- [{d.metadata['id']}] {d.page_content}" for d in docs)

    prompt = f"""Eres un experto en liquidación de siniestros de salud.

Contexto de la póliza (recuperado de la base de conocimiento):
{contexto}

Datos del siniestro:
- Prestación: {prestacion}
- Monto: {formato_clp(monto)}
- Fecha: {fecha}

Razona paso a paso, citando el ID del documento que usas:
1. ¿La prestación está cubierta por la póliza?
2. ¿El monto está dentro de los límites?
3. ¿Existe período de carencia aplicable?
4. ¿Hay algún motivo de exclusión?

Termina con este formato exacto:
Decisión final: [APROBAR / RECHAZAR / REQUIERE_REVISION_MANUAL]
Justificación: (una o dos líneas)"""

    return invocar_llm(prompt), contexto

# Ejecutar
print("=" * 60)
print("DECISIÓN CON CHAIN-OF-THOUGHT + RAG")
print("=" * 60)
decision, contexto_cot = decision_con_cot("Consulta Cardiología", 85000, "2024-03-15")
print(f"\n📋 Contexto recuperado (RAG):\n{contexto_cot}")
print(f"\n🤖 Decisión con CoT:\n{decision}")

## Celda 9 — Métricas de evaluación (LLM como juez)
*Evidencia: `09_evaluacion.png`*

In [ ]:
# ============================================================
# CELDA 9: Evaluación con métricas de calidad (LLM como juez)
# Una sola llamada evalúa las 3 métricas → ahorra cuota y tiempo
# ============================================================
import re

def extraer_metrica(texto, nombre):
    """Busca 'nombre: número' en la respuesta y lo limita al rango 1-10."""
    m = re.search(rf"{nombre}\s*[:=]\s*(\d+(?:[.,]\d+)?)", texto, re.IGNORECASE)
    if not m:
        return 5.0
    valor = float(m.group(1).replace(",", "."))
    return max(1.0, min(10.0, valor))

def evaluar_respuesta(query, contexto, respuesta, ground_truth):
    prompt = f"""Eres un evaluador de sistemas RAG. Evalúa la respuesta del sistema con 3 métricas del 1 al 10:

- FIDELIDAD: ¿la respuesta se basa SOLO en el contexto, sin inventar datos?
- RELEVANCIA: ¿la respuesta contesta directamente la consulta?
- CORRECTITUD: ¿la decisión y el motivo coinciden con la respuesta esperada?

Consulta: {query}
Contexto recuperado: {contexto}
Respuesta del sistema: {respuesta}
Respuesta esperada: {ground_truth}

Responde SOLO con estas 3 líneas, sin texto adicional:
FIDELIDAD: <número>
RELEVANCIA: <número>
CORRECTITUD: <número>"""
    texto = invocar_llm(prompt)
    return {
        "fidelidad": extraer_metrica(texto, "FIDELIDAD"),
        "relevancia": extraer_metrica(texto, "RELEVANCIA"),
        "correctitud": extraer_metrica(texto, "CORRECTITUD"),
    }

print("✅ Función de evaluación definida (3 métricas en 1 llamada)")
print("   - Fidelidad: ¿la respuesta se basa en el contexto recuperado?")
print("   - Relevancia: ¿la respuesta contesta la pregunta?")
print("   - Correctitud: ¿coincide con la respuesta esperada (ground truth)?")

## Celda 10 — Evaluación sistemática (5 consultas)
Realiza 10 llamadas al LLM (aprox. 1-2 minutos). Ejecutar **una sola vez**.

*Evidencia: `10_resultados_csv.png`*

In [ ]:
# ============================================================
# CELDA 10: Evaluación sistemática con 5 consultas (10 llamadas en total)
# ============================================================
import pandas as pd

dataset_evaluacion = [
    {"query": "¿Está cubierta una consulta cardiológica de $85.000?",
     "ground_truth": "APROBAR - dentro del límite de $100.000"},
    {"query": "¿Cubren exámenes de laboratorio por $60.000?",
     "ground_truth": "REQUIERE_REVISION - excede el límite de $50.000"},
    {"query": "¿Cubren una hospitalización de $400.000?",
     "ground_truth": "APROBAR - dentro del límite de $500.000"},
    {"query": "¿Cubren un tratamiento estético de $200.000?",
     "ground_truth": "RECHAZAR - exclusión explícita"},
    {"query": "¿Qué pasa si presento un siniestro 70 días después?",
     "ground_truth": "REQUIERE justificación escrita"},
]

resultados = []
inicio = time.time()
print("=" * 70)
print("EVALUACIÓN SISTEMÁTICA - 5 CONSULTAS (aprox. 1-2 minutos)")
print("=" * 70)

for i, caso in enumerate(dataset_evaluacion, 1):
    print(f"\n[{i}/5] Evaluando: {caso['query'][:60]}...")

    # Llamada 1: respuesta del sistema RAG
    r = consultar(caso["query"])
    time.sleep(PAUSA_ENTRE_LLAMADAS)

    # Llamada 2: evaluación de las 3 métricas
    m = evaluar_respuesta(caso["query"], r["contexto"], r["respuesta"], caso["ground_truth"])
    time.sleep(PAUSA_ENTRE_LLAMADAS)

    resultados.append({
        "consulta": caso["query"],
        "esperado": caso["ground_truth"],
        "respuesta_sistema": r["respuesta"].replace("\n", " "),
        "fuentes": ", ".join(r["fuentes"]),
        **m,
    })
    print(f"   ✓ Fidelidad: {m['fidelidad']}/10 | Relevancia: {m['relevancia']}/10 | Correctitud: {m['correctitud']}/10")

df = pd.DataFrame(resultados)
print("\n" + "=" * 70)
print("RESULTADOS FINALES")
print("=" * 70)
print(df[["consulta", "fuentes", "fidelidad", "relevancia", "correctitud"]].to_string(index=False))
print(f"\n📊 Fidelidad promedio:   {df['fidelidad'].mean():.2f}/10")
print(f"📊 Relevancia promedio:  {df['relevancia'].mean():.2f}/10")
print(f"📊 Correctitud promedio: {df['correctitud'].mean():.2f}/10")
print(f"⏱️ Tiempo total: {time.time() - inicio:.0f} segundos")

# Exportar CSV (utf-8-sig para que Excel muestre bien los tildes)
df.to_csv("resultados_evaluacion.csv", index=False, encoding="utf-8-sig")
print("\n✅ CSV exportado: resultados_evaluacion.csv")

## Celda 11 — Gráfico de resultados
*Evidencia: `11_grafico.png`*

In [ ]:
# ============================================================
# CELDA 11: Visualización de resultados
# ============================================================
import matplotlib.pyplot as plt

metricas = ["fidelidad", "relevancia", "correctitud"]
nombres = ["Fidelidad", "Relevancia", "Correctitud"]
colores = ["#1A3A5C", "#00A86B", "#E8A33D"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Barras comparativas por consulta
x = list(range(1, len(df) + 1))
width = 0.25
for j, (m, nombre, color) in enumerate(zip(metricas, nombres, colores)):
    axes[0].bar([i + (j - 1) * width for i in x], df[m], width, label=nombre, color=color)
axes[0].set_xlabel("Consulta")
axes[0].set_ylabel("Puntuación (1-10)")
axes[0].set_title("Métricas por Consulta")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"C{i}" for i in x])
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)
axes[0].set_ylim(0, 11)

# Gráfico 2: Promedios
promedios = [df[m].mean() for m in metricas]
axes[1].bar(nombres, promedios, color=colores)
axes[1].set_ylabel("Puntuación promedio")
axes[1].set_title(f"Promedios Generales ({MODELO_LLM})")
axes[1].set_ylim(0, 11)
for i, v in enumerate(promedios):
    axes[1].text(i, v + 0.2, f"{v:.2f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("grafico_evaluacion.png", dpi=150, bbox_inches="tight")
plt.show()

print("✅ Gráfico guardado: grafico_evaluacion.png")

## Celda 12 — Descargar archivos generados
Subir `resultados_evaluacion.csv` y `grafico_evaluacion.png` a la carpeta `/resultados` del repositorio.

In [ ]:
# ============================================================
# CELDA 12: Descargar archivos para el repositorio
# ============================================================
from google.colab import files

print("📥 Descargando archivos generados...")
files.download("resultados_evaluacion.csv")
files.download("grafico_evaluacion.png")
print("✅ Archivos descargados")